# 49 â€” Wide & Deep MLP

Wide & Deep neural network:
- **Wide** component: linear model on Morgan FP (2048-dim) â€” memorizes scaffold-specific activity
- **Deep** component: MLP on RDKit descriptors (217-dim) â€” generalizes via physicochemical space
- **Cross-attention variant**: Wide (Morgan FP) as Query, Deep (RDKit) as Key/Value

**Primary metric:** RAE (lower is better). Current best OOF RAE: 0.5281.

In [1]:
import os as _os
_torch_lib = r"d:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\torch\lib"
if _os.path.exists(_torch_lib):
    _os.add_dll_directory(_torch_lib)

import sys, os
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute, morgan, rdkit_desc
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                   reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load data â€” separate Morgan FP and RDKit descriptors

In [2]:
tr = load_train()
te = load_test()
print(f"Train: {len(tr)} | Test: {len(te)}")

tr['scaffold'] = tr['smiles'].apply(bemis_murcko)
y_tr = tr['pec50'].values.astype(np.float32)

# Separate feature sets
print("Computing Morgan FP...")
X_morgan_tr = impute(morgan(tr['smiles'].tolist()).astype(np.float32))   # (N, 2048)
X_morgan_te = impute(morgan(te['smiles'].tolist()).astype(np.float32))   # (M, 2048)

print("Computing RDKit descriptors...")
X_rdkit_tr = impute(rdkit_desc(tr['smiles'].tolist()).astype(np.float32))  # (N, ~217)
X_rdkit_te = impute(rdkit_desc(te['smiles'].tolist()).astype(np.float32))  # (M, ~217)

d_wide = X_morgan_tr.shape[1]
d_deep = X_rdkit_tr.shape[1]
print(f"Wide (Morgan FP): {d_wide} | Deep (RDKit): {d_deep}")

Train: 4139 | Test: 513


Computing Morgan FP...


Computing RDKit descriptors...


Wide (Morgan FP): 2048 | Deep (RDKit): 217


## 2. Wide & Deep architecture

In [3]:
class WideDeep(nn.Module):
    """Wide & Deep: Wide = linear on Morgan FP, Deep = MLP on RDKit descriptors."""
    def __init__(self, d_wide=2048, d_deep=217, d_hidden=256):
        super().__init__()
        # Wide: linear model on Morgan FP (memorization)
        self.wide = nn.Linear(d_wide, 1, bias=True)
        # Deep: MLP on RDKit descriptors (generalization)
        self.deep = nn.Sequential(
            nn.Linear(d_deep, d_hidden), nn.BatchNorm1d(d_hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_hidden, d_hidden), nn.BatchNorm1d(d_hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_hidden, 64)
        )
        # Joint output layer
        self.output = nn.Linear(1 + 64, 1)

    def forward(self, x_wide, x_deep):
        w = self.wide(x_wide)       # (B, 1)
        d = self.deep(x_deep)       # (B, 64)
        return self.output(torch.cat([w, d], dim=1)).squeeze(-1)


class WideDeepCrossAttn(nn.Module):
    """Wide & Deep with cross-attention between Morgan FP and RDKit spaces.

    Morgan FP (projected) as Query; RDKit embeddings as Key and Value.
    This allows the model to selectively attend to physicochemical
    properties that are most relevant for each specific FP pattern.
    """
    def __init__(self, d_wide=2048, d_deep=217, d_hidden=256, n_heads=4):
        super().__init__()
        self.d_hidden = d_hidden

        # Compress Morgan FP to d_hidden
        self.wide_proj = nn.Sequential(
            nn.Linear(d_wide, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
        )
        # Compress RDKit descriptors to d_hidden
        self.deep_proj = nn.Sequential(
            nn.Linear(d_deep, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
        )

        # Cross-attention: Wide as Q, Deep as K,V
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_hidden, num_heads=n_heads,
            dropout=0.1, batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(d_hidden)

        # Also keep the plain Wide output as residual
        self.wide_reg = nn.Linear(d_wide, 1, bias=True)

        # Final MLP on concatenation of cross-attn output + wide regression
        self.output = nn.Sequential(
            nn.Linear(d_hidden + 1, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x_wide, x_deep):
        q = self.wide_proj(x_wide).unsqueeze(1)   # (B, 1, d_hidden) â€” query
        kv = self.deep_proj(x_deep).unsqueeze(1)  # (B, 1, d_hidden) â€” key & value

        # Cross-attention: how much should each FP pattern attend to physchem?
        attn_out, _ = self.cross_attn(q, kv, kv)  # (B, 1, d_hidden)
        attn_out = self.attn_norm(attn_out.squeeze(1) + q.squeeze(1))  # residual

        wide_signal = self.wide_reg(x_wide)  # (B, 1)
        combined = torch.cat([attn_out, wide_signal], dim=-1)  # (B, d_hidden + 1)
        return self.output(combined).squeeze(-1)


print("WideDeep and WideDeepCrossAttn defined.")
wd = WideDeep(d_wide=d_wide, d_deep=d_deep)
wdca = WideDeepCrossAttn(d_wide=d_wide, d_deep=d_deep)
print(f"WideDeep params: {sum(p.numel() for p in wd.parameters()):,}")
print(f"WideDeepCrossAttn params: {sum(p.numel() for p in wdca.parameters()):,}")

WideDeep and WideDeepCrossAttn defined.
WideDeep params: 141,187
WideDeepCrossAttn params: 880,258


## 3. Training utilities

In [4]:
def train_wide_deep(
    X_wide_train, X_deep_train, y_train,
    model_cls=WideDeep,
    n_epochs=200, batch_size=128, lr=2e-3,
    device=DEVICE,
):
    # Wide (Morgan FP) stays as-is (binary) â€” no StandardScaler
    # Deep (RDKit) gets StandardScaler
    deep_scaler = StandardScaler()
    X_deep_sc = deep_scaler.fit_transform(X_deep_train).astype(np.float32)
    X_wide_f = X_wide_train.astype(np.float32)

    d_wide = X_wide_f.shape[1]
    d_deep = X_deep_sc.shape[1]

    model = model_cls(d_wide=d_wide, d_deep=d_deep).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    loss_fn = nn.HuberLoss(delta=1.0)

    Xw_t = torch.tensor(X_wide_f, dtype=torch.float32)
    Xd_t = torch.tensor(X_deep_sc, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32)
    ds = TensorDataset(Xw_t, Xd_t, y_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    best_loss = float('inf')
    best_state = None
    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for xw_b, xd_b, y_b in loader:
            xw_b, xd_b, y_b = xw_b.to(device), xd_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            pred = model(xw_b, xd_b)
            loss = loss_fn(pred, y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 50 == 0:
            print(f"    Epoch {epoch+1}/{n_epochs}  loss={epoch_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, deep_scaler


@torch.no_grad()
def predict_wide_deep(model, deep_scaler, X_wide, X_deep, batch_size=512, device=DEVICE):
    model.eval()
    X_wide_f = X_wide.astype(np.float32)
    X_deep_sc = deep_scaler.transform(X_deep).astype(np.float32)
    Xw_t = torch.tensor(X_wide_f, dtype=torch.float32)
    Xd_t = torch.tensor(X_deep_sc, dtype=torch.float32)
    preds = []
    for i in range(0, len(Xw_t), batch_size):
        xw_b = Xw_t[i:i+batch_size].to(device)
        xd_b = Xd_t[i:i+batch_size].to(device)
        preds.append(model(xw_b, xd_b).cpu().numpy())
    return np.concatenate(preds)

print("Training utilities defined.")

Training utilities defined.


## 4. Scaffold 5-fold CV â€” compare Wide-only, Deep-only, Wide+Deep, CrossAttn variants

In [5]:
# Also define Wide-only and Deep-only wrappers for ablation
class WideOnly(nn.Module):
    def __init__(self, d_wide=2048, d_deep=217, **kw):
        super().__init__()
        self.wide = nn.Linear(d_wide, 1)

    def forward(self, x_wide, x_deep):
        return self.wide(x_wide).squeeze(-1)


class DeepOnly(nn.Module):
    def __init__(self, d_wide=2048, d_deep=217, d_hidden=256):
        super().__init__()
        self.deep = nn.Sequential(
            nn.Linear(d_deep, d_hidden), nn.BatchNorm1d(d_hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_hidden, d_hidden), nn.BatchNorm1d(d_hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_hidden, 1)
        )

    def forward(self, x_wide, x_deep):
        return self.deep(x_deep).squeeze(-1)


splits = scaffold_kfold_indices(tr['scaffold'], n_splits=5)

variants = [
    ('WideOnly',    WideOnly),
    ('DeepOnly',    DeepOnly),
    ('WideDeep',    WideDeep),
    ('CrossAttn',   WideDeepCrossAttn),
]

oof_by_variant = {name: np.full(len(tr), np.nan) for name, _ in variants}

for fold, (tr_idx, val_idx) in enumerate(splits):
    print(f"\n=== Fold {fold+1}/5 â€” train={len(tr_idx)}, val={len(val_idx)} ===")
    for var_name, var_cls in variants:
        model, scaler = train_wide_deep(
            X_morgan_tr[tr_idx], X_rdkit_tr[tr_idx], y_tr[tr_idx],
            model_cls=var_cls, n_epochs=200, batch_size=128, lr=2e-3,
        )
        val_preds = predict_wide_deep(model, scaler, X_morgan_tr[val_idx], X_rdkit_tr[val_idx])
        fold_rae = rae(y_tr[val_idx], val_preds)
        oof_by_variant[var_name][val_idx] = val_preds
        print(f"  {var_name:15s}  fold RAE: {fold_rae:.4f}")

print("\n=== OOF RAE by variant ===")
for name, oof in oof_by_variant.items():
    print(f"  {name:15s}: {rae(y_tr, oof):.4f}")


=== Fold 1/5 â€” train=3311, val=828 ===


    Epoch 50/200  loss=3.1415


    Epoch 100/200  loss=2.7012


    Epoch 150/200  loss=2.5720


    Epoch 200/200  loss=2.5418
  WideOnly         fold RAE: 0.7553


...

    Epoch 200/200  loss=0.4688
  WideDeep         fold RAE: 0.7755


    Epoch 50/200  loss=0.6802


    Epoch 100/200  loss=0.4400


    Epoch 150/200  loss=0.3610


    Epoch 200/200  loss=0.3362
  CrossAttn        fold RAE: 0.6480

=== OOF RAE by variant ===
  WideOnly       : 0.8623
  DeepOnly       : 1.0946
  WideDeep       : 0.7857
  CrossAttn      : 0.5911


In [6]:
# Select best variant based on OOF RAE
best_variant_name = min(oof_by_variant, key=lambda n: rae(y_tr, oof_by_variant[n]))
best_cls = dict(variants)[best_variant_name]
oof_preds = oof_by_variant[best_variant_name]
oof_rae = rae(y_tr, oof_preds)
print(f"Best variant: {best_variant_name}  OOF RAE: {oof_rae:.4f}")

Best variant: CrossAttn  OOF RAE: 0.5911


## 5. Final model â€” train on all data, predict test

In [7]:
print(f"Training final {best_variant_name} on all training data...")
final_model, final_scaler = train_wide_deep(
    X_morgan_tr, X_rdkit_tr, y_tr,
    model_cls=best_cls, n_epochs=200, batch_size=128, lr=2e-3,
)

test_preds = predict_wide_deep(final_model, final_scaler, X_morgan_te, X_rdkit_te)

y_lo = y_tr.min() - 0.5
y_hi = y_tr.max() + 0.5
test_preds = np.clip(test_preds, y_lo, y_hi)
print(f"Test pred range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")

Training final CrossAttn on all training data...


    Epoch 50/200  loss=0.8653


    Epoch 100/200  loss=0.5580


    Epoch 150/200  loss=0.4302


    Epoch 200/200  loss=0.4010
Test pred range: [2.033, 6.313]


In [8]:
# Save OOF
np.save(DATA_PROCESSED / 'oof_wide_deep.npy', oof_preds)
print("Saved OOF predictions.")

# Save submission
sub = pd.DataFrame({'Molecule Name': te['name'], 'pEC50': test_preds})
out_path = SUBMISSIONS / '49_wide_deep_mlp.csv'
sub.to_csv(out_path, index=False)
print(f"Saved submission to {out_path}")
print(sub.head())
print(f"\nFinal OOF RAE ({best_variant_name}): {oof_rae:.4f}")
print("\nFull variant OOF RAE summary:")
for name, oof in oof_by_variant.items():
    print(f"  {name:15s}: {rae(y_tr, oof):.4f}")

Saved OOF predictions.
Saved submission to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\49_wide_deep_mlp.csv
    Molecule Name     pEC50
0  OADMET-0006617  4.170559
1  OADMET-0006616  3.831614
2  OADMET-0006615  5.442133
3  OADMET-0006614  5.622607
4  OADMET-0006613  4.818431

Final OOF RAE (CrossAttn): 0.5911

Full variant OOF RAE summary:
  WideOnly       : 0.8623
  DeepOnly       : 1.0946
  WideDeep       : 0.7857
  CrossAttn      : 0.5911
